### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
from datetime import datetime
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/11 19:10:59 WARN Utils: Your hostname, LAPTOP-UGSS09T8 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/11 19:10:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 19:11:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

In [2]:
schema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

folder_stream = "tugas10_stream"
for nama_folder in [folder_stream, "tugas10_output", "tugas10_checkpoint"]:
    shutil.rmtree(nama_folder, ignore_errors=True)

os.makedirs(folder_stream, exist_ok=True)

print("Folder streaming siap:", folder_stream)

Folder streaming siap: tugas10_stream


In [3]:
df_stream = spark.readStream \
    .format("csv") \
    .schema(schema_pesanan) \
    .option("header", True) \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .load(folder_stream)

print("df_stream.isStreaming:", df_stream.isStreaming)

df_stream.isStreaming: True


**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

In [4]:
pesanan_dengan_total = df_stream.withColumn(
    "total_harga", col("qty") * col("harga_satuan")
)

hasil_per_cabang = pesanan_dengan_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

print("Agregasi per cabang berhasil dibuat.")

Agregasi per cabang berhasil dibuat.


In [5]:
query_cabang = hasil_per_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi cabang aktif:", query_cabang.isActive)

Query agregasi cabang aktif: True


**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

In [6]:
stream_untuk_window = spark.readStream \
    .format("csv") \
    .schema(schema_pesanan) \
    .option("header", True) \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .load(folder_stream)

stream_untuk_window = stream_untuk_window.withColumn(
    "total_harga", col("qty") * col("harga_satuan")
)

hasil_window = stream_untuk_window.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

query_window = hasil_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query window 2 menit aktif:", query_window.isActive)

Query window 2 menit aktif: True


**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.


In [7]:
# Batch 1: pesanan awal dari Magelang dan Solo
waktu_batch = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

data_batch1 = {
    "pesanan_id": ["KN01", "KN02", "KN03"],
    "nama_cabang": ["Magelang", "Solo", "Magelang"],
    "nama_menu": ["Jus Alpukat", "Caramel Machiato", "Pisang Coklat"],
    "qty": [2, 1, 2],
    "harga_satuan": [12000, 20000, 15000],
    "waktu_pesan": [waktu_batch] * 3
}
pd.DataFrame(data_batch1).to_csv(f"{folder_stream}/pesanan_1.csv", index=False)

print("Batch pertama sudah dikirim.")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

Batch pertama sudah dikirim.


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|           54000|             2|
|       Solo|           20000|             1|
+-----------+----------------+--------------+



In [8]:
# Batch 2: menambahkan pesanan pada cabang yang sudah ada dan cabang baru
waktu_batch = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

data_batch2 = {
    "pesanan_id": ["KN04", "KN05"],
    "nama_cabang": ["Magelang", "Semarang"],
    "nama_menu": ["Matcha", "Jus Alpukat"],
    "qty": [1, 3],
    "harga_satuan": [22000, 12000],
    "waktu_pesan": [waktu_batch] * 2
}
pd.DataFrame(data_batch2).to_csv(f"{folder_stream}/pesanan_2.csv", index=False)

print("Batch kedua sudah dikirim.")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

Batch kedua sudah dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|           76000|             3|
|   Semarang|           36000|             1|
|       Solo|           20000|             1|
+-----------+----------------+--------------+



In [9]:
# Batch 3: menambahkan data dari beberapa cabang
waktu_batch = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

data_batch3 = {
    "pesanan_id": ["KN06", "KN07", "KN08"],
    "nama_cabang": ["Solo", "Yogyakarta", "Semarang"],
    "nama_menu": ["Pisang Bakar", "Jus Alpukat", "Caramel Machiato"],
    "qty": [2, 2, 1],
    "harga_satuan": [15000, 12000, 20000],
    "waktu_pesan": [waktu_batch] * 3
}
pd.DataFrame(data_batch3).to_csv(f"{folder_stream}/pesanan_3.csv", index=False)

print("Batch ketiga sudah dikirim.")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

Batch ketiga sudah dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|           76000|             3|
|   Semarang|           56000|             2|
|       Solo|           50000|             2|
| Yogyakarta|           24000|             1|
+-----------+----------------+--------------+



**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

In [10]:
query_file = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

In [11]:
query_file = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Query penyimpanan Parquet aktif:", query_file.isActive)

[Stage 15:>                                                         (0 + 3) / 3]

Query penyimpanan Parquet aktif: True


In [12]:
# Batch keempat dikirim setelah query Parquet berjalan
waktu_batch = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

data_batch4 = {
    "pesanan_id": ["KN09"],
    "nama_cabang": ["Yogyakarta"],
    "nama_menu": ["Es Cokelat"],
    "qty": [2],
    "harga_satuan": [17000],
    "waktu_pesan": [waktu_batch]
}
pd.DataFrame(data_batch4).to_csv(f"{folder_stream}/pesanan_4.csv", index=False)

print("Batch keempat dikirim untuk pengujian file sink.")
time.sleep(10)

Batch keempat dikirim untuk pengujian file sink.


In [13]:
!ls tugas10_output

_spark_metadata
part-00000-63910808-f12b-4224-99d2-5d4c5aeb3c4a-c000.snappy.parquet
part-00000-ccf621c8-927b-4ea5-959c-f9f58878bad5-c000.snappy.parquet
part-00001-e2b3f2d5-4d69-492f-a2ef-48969504f52b-c000.snappy.parquet
part-00002-7bfcee1c-f079-4bcd-a1de-803130a76442-c000.snappy.parquet


In [14]:
# Memeriksa file hasil streaming dan membacanya kembali
print("Daftar file dalam folder Parquet:")
print(os.listdir("tugas10_output"))

data_parquet = spark.read.parquet("tugas10_output")
print("Isi data yang berhasil dibaca dari Parquet:")
data_parquet.show(truncate=False)

print("Jumlah data tersimpan:", data_parquet.count())

Daftar file dalam folder Parquet:
['part-00000-63910808-f12b-4224-99d2-5d4c5aeb3c4a-c000.snappy.parquet', '.part-00000-ccf621c8-927b-4ea5-959c-f9f58878bad5-c000.snappy.parquet.crc', '.part-00000-63910808-f12b-4224-99d2-5d4c5aeb3c4a-c000.snappy.parquet.crc', 'part-00002-7bfcee1c-f079-4bcd-a1de-803130a76442-c000.snappy.parquet', '.part-00002-7bfcee1c-f079-4bcd-a1de-803130a76442-c000.snappy.parquet.crc', 'part-00000-ccf621c8-927b-4ea5-959c-f9f58878bad5-c000.snappy.parquet', 'part-00001-e2b3f2d5-4d69-492f-a2ef-48969504f52b-c000.snappy.parquet', '.part-00001-e2b3f2d5-4d69-492f-a2ef-48969504f52b-c000.snappy.parquet.crc', '_spark_metadata']
Isi data yang berhasil dibaca dari Parquet:
+----------+-----------+----------------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu       |qty|harga_satuan|waktu_pesan        |
+----------+-----------+----------------+---+------------+-------------------+
|KN01      |Magelang   |Jus Alpukat     |2  |12000       |2026-11-11 19:11:08

In [15]:
query_cabang.stop()
query_window.stop()
query_file.stop()

spark.stop()
print("Semua query dihentikan dan SparkSession ditutup.")

Semua query dihentikan dan SparkSession ditutup.
